# FlashRank-Pro Training Pipeline
---
**Built by [Eulogik](https://eulogik.com)** · [GitHub](https://github.com/eulogik/flashrank-pro)

Resumable notebook for Colab T4. Each stage saves to Google Drive.
If the session drops, just Runtime → Run all — it picks up where it left off.

**Before you start:**
- Go to https://colab.research.google.com/drive/ and upload this notebook
- Runtime → Change runtime type → T4 GPU
- (Optional) Set `OPENAI_API_KEY` secret in the 🔑 Secrets panel if you want Stage 1

In [ ]:
# ============================================================
# SETUP: Install dependencies, mount Drive, clone repo
# ============================================================
import os, sys, json, shutil, glob, subprocess, time, warnings
from pathlib import Path

DRIVE_MOUNT = "/content/drive"
DRIVE_ROOT  = f"{DRIVE_MOUNT}/MyDrive/flashrank-pro"
LOCAL_ROOT  = "/content/flashrank-pro"
GIT_REPO    = "https://github.com/eulogik/flashrank-pro.git"

print("⚡ FlashRank-Pro Training Pipeline")
print("=" * 50)

# ---- 1. Install dependencies ----
DEPS_INSTALLED = Path("/content/.fpdeps")
if not DEPS_INSTALLED.exists():
    print("\n[1/6] Installing dependencies...")
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    !pip install -q transformers sentence-transformers accelerate datasets fire tqdm peft
    !pip install -q openai huggingface-hub
    DEPS_INSTALLED.touch()
    print("    ✅ Dependencies installed")
else:
    print("\n[1/6] Dependencies already installed, skipping")

# ---- 2. Mount Google Drive ----
print("\n[2/6] Mounting Google Drive...")
from google.colab import drive
drive.mount(DRIVE_MOUNT, force_remount=False)
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"    ✅ Drive mounted at {DRIVE_ROOT}")

# ---- 3. Clone or pull repo ----
print("\n[3/6] Cloning repository...")
if not os.path.exists(LOCAL_ROOT):
    !git clone {GIT_REPO} {LOCAL_ROOT}
    print("    ✅ Fresh clone")
else:
    %cd {LOCAL_ROOT}
    !git pull --ff-only
    %cd /content
    print("    ✅ Repo updated")

%cd {LOCAL_ROOT}
sys.path.insert(0, LOCAL_ROOT)

# ---- 4. Restore previous progress from Drive ----
print("\n[4/6] Restoring previous progress from Drive...")
for item in ["data", "models"]:
    local_p = Path(LOCAL_ROOT) / item
    drive_p = Path(DRIVE_ROOT) / item
    if drive_p.exists() and not local_p.exists():
        shutil.copytree(str(drive_p), str(local_p), symlinks=True, ignore_dangling_symlinks=True)
        print(f"    ✅ Restored {item}/ from Drive")
    elif local_p.exists():
        print(f"    ✅ {item}/ already present locally")
    else:
        os.makedirs(local_p, exist_ok=True)
        print(f"    📁 Created {item}/ (empty)")

# ---- 5. Check GPU ----
print("\n[5/6] Checking GPU...")
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"    ✅ {gpu} ({mem:.0f} GB VRAM)")
else:
    print("    ⚠️  No GPU found — expect slow training")

# ---- 6. Save current status ----
print("\n[6/6] Setup complete")
status = {
    "stages": {
        "data": os.path.exists(f"{LOCAL_ROOT}/data/synthetic_training_data.jsonl"),
        "kd_en": os.path.exists(f"{LOCAL_ROOT}/models/flashrank-pro-base-kd-en"),
        "kd_multi": os.path.exists(f"{LOCAL_ROOT}/models/flashrank-pro-base-kd-multilingual"),
        "rl": os.path.exists(f"{LOCAL_ROOT}/models/flashrank-pro-base-rl"),
        "merged": os.path.exists(f"{LOCAL_ROOT}/models/flashrank-pro-merged"),
    }
}
print(json.dumps(status, indent=2))
print("=" * 50)

---
## Stage 1: Generate synthetic training data
**Cost:** ~$7.50 (GPT-4o-mini) | **Hardware:** Any | **Time:** ~1-2h

Set `OPENAI_API_KEY` in Secrets panel (🔑) before running this cell.
If no API key is set, this stage is skipped and you can use default data later.

In [ ]:
# ============================================================
# STAGE 1: Generate synthetic training data
# ============================================================
data_done = Path(f"{LOCAL_ROOT}/data/synthetic_training_data.jsonl")

if data_done.exists():
    lines = len(open(data_done).readlines())
    print(f"✅ Stage 1 already complete — {lines} examples in data/synthetic_training_data.jsonl")
else:
    from google.colab import userdata
    api_key = None
    try:
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass

    if not api_key:
        print("⚠️  No OPENAI_API_KEY found. Stage 1 requires it.")
        print("   Add it: 🔑 Secrets panel → Add new secret → OPENAI_API_KEY")
        print("   Then re-run this cell. Skipping for now.")
    else:
        print("▶️  Running Stage 1: Generate synthetic data...")
        !python training/01_generate_synthetic_data.py \
            --output_path data/synthetic_training_data.jsonl \
            --corpus_name sentence-transformers/gooaq \
            --n_queries 50000 \
            --n_negatives 4

        # Save to Drive
        !cp -r data/synthetic_training_data.jsonl {DRIVE_ROOT}/data/
        print(f"✅ Stage 1 complete. Saved to Drive and local.")

---
## Stage 2: Knowledge Distillation
**Hardware:** T4 (16GB) | **Time:** ~3h (base, full FT), ~3h (large, LoRA)

Trains ModernBERT using soft labels from teacher reranker.
Adjust `MODEL_SIZE` below for `base` or `large`.

> ⏱️ This takes ~3 hours. If session drops, re-run — it will resume.

In [ ]:
# ============================================================
# STAGE 2: Knowledge Distillation
# ============================================================
MODEL_SIZE = "base"  # CHANGE to "large" for ModernBERT-large
MODEL_NAME = f"answerdotai/ModernBERT-{MODEL_SIZE}"
KD_OUTPUT  = f"models/flashrank-pro-{MODEL_SIZE}-kd-en"
DRIVE_KD   = f"{DRIVE_ROOT}/models/flashrank-pro-{MODEL_SIZE}-kd-en"

if os.path.exists(f"{LOCAL_ROOT}/{KD_OUTPUT}"):
    print(f"✅ Stage 2 ({MODEL_SIZE}) already complete — model at {KD_OUTPUT}")
else:
    # Check data exists
    if not os.path.exists("data/synthetic_training_data.jsonl"):
        print("❌ data/synthetic_training_data.jsonl not found. Run Stage 1 first.")
    else:
        LORA_FLAG = "--use_lora" if MODEL_SIZE == "large" else ""
        BATCH_SIZE = 8 if MODEL_SIZE == "base" else 4

        print(f"▶️  Running Stage 2: KD for {MODEL_NAME}")
        print(f"    LoRA: {'yes' if LORA_FLAG else 'no'} | Batch: {BATCH_SIZE} | Output: {KD_OUTPUT}")

        !python training/02_knowledge_distillation.py \
            --model_name {MODEL_NAME} \
            --data_path data/synthetic_training_data.jsonl \
            --output_dir {KD_OUTPUT} \
            --batch_size {BATCH_SIZE} \
            --num_epochs 3 \
            --learning_rate 2e-5 \
            {LORA_FLAG} \
            --lora_r 16

        # Save to Drive immediately
        os.makedirs(os.path.dirname(DRIVE_KD), exist_ok=True)
        if os.path.exists(f"{LOCAL_ROOT}/{KD_OUTPUT}"):
            !cp -r {KD_OUTPUT} {DRIVE_KD}
            print(f"✅ Stage 2 complete. Saved to Drive.")
        else:
            print("⚠️  Model output not found. Check logs above for errors.")

---
## Stage 2b: Multilingual Distillation (optional)
**Time:** ~2h | Improves cross-lingual performance.

Same as Stage 2 but uses a multilingual corpus. Skip if not needed.

In [ ]:
# ============================================================
# STAGE 2b: Multilingual Knowledge Distillation (optional)
# ============================================================
KD_MULTI_OUTPUT = f"models/flashrank-pro-{MODEL_SIZE}-kd-multilingual"
DRIVE_KD_MULTI  = f"{DRIVE_ROOT}/models/flashrank-pro-{MODEL_SIZE}-kd-multilingual"

if os.path.exists(f"{LOCAL_ROOT}/{KD_MULTI_OUTPUT}"):
    print(f"✅ Stage 2b already complete")
else:
    print("ℹ️  Skipping multilingual KD. Uses same data pipeline;")
    print("   requires a multilingual corpus (e.g., mmarco). Skip if not needed.")
    # Uncomment below to run with multilingual data
    # !python training/02_knowledge_distillation.py \
    #     --model_name answerdotai/ModernBERT-{MODEL_SIZE} \
    #     --data_path data/synthetic_training_data.jsonl \
    #     --output_dir {KD_MULTI_OUTPUT} \
    #     --batch_size 8 --num_epochs 2
    # !cp -r {KD_MULTI_OUTPUT} {DRIVE_KD_MULTI}

---
## Stage 3: GRPO Reinforcement Learning
**Hardware:** T4 (16GB) | **Time:** ~1-2h

RL fine-tuning with GRPO prompt warmup and fine-grained scoring.
Requires Stage 2 KD model as starting point.

In [ ]:
# ============================================================
# STAGE 3: GRPO RL Fine-Tuning
# ============================================================
RL_INPUT   = KD_OUTPUT
RL_OUTPUT  = f"models/flashrank-pro-{MODEL_SIZE}-rl"
DRIVE_RL   = f"{DRIVE_ROOT}/models/flashrank-pro-{MODEL_SIZE}-rl"

if os.path.exists(f"{LOCAL_ROOT}/{RL_OUTPUT}"):
    print(f"✅ Stage 3 already complete — model at {RL_OUTPUT}")
else:
    if not os.path.exists(f"{LOCAL_ROOT}/{RL_INPUT}"):
        print(f"❌ {RL_INPUT} not found. Run Stage 2 first.")
    else:
        print(f"▶️  Running Stage 3: GRPO RL on {RL_INPUT}")

        !python training/03_grpo_rl.py \
            --model_path {RL_INPUT} \
            --data_path data/synthetic_training_data.jsonl \
            --output_dir {RL_OUTPUT} \
            --batch_size 4 \
            --k_samples 8 \
            --num_epochs 1

        if os.path.exists(f"{LOCAL_ROOT}/{RL_OUTPUT}"):
            os.makedirs(os.path.dirname(DRIVE_RL), exist_ok=True)
            !cp -r {RL_OUTPUT} {DRIVE_RL}
            print(f"✅ Stage 3 complete. Saved to Drive.")
        else:
            print("⚠️  RL model output not found.")

---
## Stage 4: SLERP Merge & Evaluation
**Time:** ~5min | Merges KD + RL checkpoints into final model.

Runs on CPU. Can also run locally on your Mac.

In [ ]:
# ============================================================
# STAGE 4: SLERP Merge
# ============================================================
MERGED_OUTPUT = f"models/flashrank-pro-{MODEL_SIZE}-merged"
DRIVE_MERGED  = f"{DRIVE_ROOT}/models/flashrank-pro-{MODEL_SIZE}-merged"

if os.path.exists(f"{LOCAL_ROOT}/{MERGED_OUTPUT}"):
    print(f"✅ Stage 4 already complete — model at {MERGED_OUTPUT}")
else:
    # Build slerp config from available checkpoints
    available = []
    for ckpt in [KD_OUTPUT, KD_MULTI_OUTPUT, RL_OUTPUT]:
        if os.path.exists(f"{LOCAL_ROOT}/{ckpt}"):
            available.append(ckpt)

    if len(available) < 2:
        print(f"⚠️  Need at least 2 checkpoints to merge. Found: {available}")
        print("   Copying single checkpoint as merged output...")
        if available:
            shutil.copytree(f"{LOCAL_ROOT}/{available[0]}", f"{LOCAL_ROOT}/{MERGED_OUTPUT}")
            print(f"    Copied {available[0]} → {MERGED_OUTPUT}")
    else:
        weights = [1.0 / len(available)] * len(available)
        slerp_cfg = {"checkpoints": [f"{LOCAL_ROOT}/{c}" for c in available], "weights": weights}
        with open("configs/slerp_config.json", "w") as f:
            json.dump(slerp_cfg, f)
        print(f"▶️  Merging {len(available)} checkpoints via SLERP: {available}")

        !python training/04_slerp_merge.py \
            --config_path configs/slerp_config.json \
            --output_path {MERGED_OUTPUT}

    if os.path.exists(f"{LOCAL_ROOT}/{MERGED_OUTPUT}"):
        os.makedirs(os.path.dirname(DRIVE_MERGED), exist_ok=True)
        !cp -r {MERGED_OUTPUT} {DRIVE_MERGED}
        print(f"✅ Stage 4 complete. Merged model at {MERGED_OUTPUT}")

---
## Evaluation (quick check)
Runs a lightweight sanity check on the merged model.

In [ ]:
# ============================================================
# QUICK EVALUATION
# ============================================================
if os.path.exists(f"{LOCAL_ROOT}/{MERGED_OUTPUT}"):
    print("▶️  Quick sanity check...")

    from flashrank_pro import Reranker

    reranker = Reranker(f"{LOCAL_ROOT}/{MERGED_OUTPUT}", device="cuda" if torch.cuda.is_available() else "cpu")

    query = "how to train a neural network"
    docs = [
        "Training neural networks requires backpropagation through the computational graph.",
        "Python is a high-level programming language created by Guido van Rossum.",
        "The ancient Roman Empire spanned three continents and lasted over 500 years.",
        "Gradient descent optimizes loss functions by iteratively updating parameters.",
        "Neural networks learn by adjusting weights based on error gradients.",
    ]

    results = reranker.rerank(query, docs)
    print(f"\nQuery: {query}\n")
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['score']:.4f}] {r['text'][:80]}...")

    # Verify ranking: ML docs should rank above history/python
    top = [r["text"] for r in results]
    ml_terms = ["neural", "gradient", "backpropagation", "weights", "loss"]
    has_ml_top = any(t in top[0].lower() for t in ml_terms)
    print(f"\n{'✅ ML doc ranked first' if has_ml_top else '⚠️ ML doc not first'}")

    # Benchmarks
    print("\n--- Full benchmarks ---")
    print("Run: python scripts/evaluate.py --model_path " + MERGED_OUTPUT + " --benchmark beir")
else:
    print("⚠️  No merged model found. Run Stages 1-4 first.")

---
## Deploy to HuggingFace
Set `HF_TOKEN` in Secrets (🔑) before running.
Get your token at https://huggingface.co/settings/tokens

In [ ]:
# ============================================================
# DEPLOY
# ============================================================
if os.path.exists(f"{LOCAL_ROOT}/{MERGED_OUTPUT}"):
    from google.colab import userdata
    hf_token = None
    try:
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

    if hf_token:
        repo_id = f"eulogik/flashrank-pro-{MODEL_SIZE}"
        print(f"▶️  Deploying to HuggingFace: {repo_id}")
        !python scripts/deploy_to_huggingface.py \
            --model_path {MERGED_OUTPUT} \
            --repo_id {repo_id}
        print(f"✅ Model at https://huggingface.co/{repo_id}")
    else:
        print("⚠️  No HF_TOKEN set. Add in Secrets panel → HF_TOKEN")
        print("   Then re-run this cell.")
else:
    print("⚠️  No merged model. Run all stages first.")

---
## Status Summary

| Stage | Status | Drive Path |
|-------|--------|------------|
| 1. Data | ✅/⬜ | Drive/flashrank-pro/data/ |
| 2. KD English | ✅/⬜ | Drive/flashrank-pro/models/*-kd-en/ |
| 2b. KD Multilingual | ✅/⬜ | Drive/flashrank-pro/models/*-kd-multilingual/ |
| 3. GRPO RL | ✅/⬜ | Drive/flashrank-pro/models/*-rl/ |
| 4. Merge | ✅/⬜ | Drive/flashrank-pro/models/*-merged/ |

All data and models persist in `MyDrive/flashrank-pro/` on Google Drive.
If your session disconnects, go to **Runtime → Run all** — it resumes automatically.